In [30]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
import networkx as nx
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
matplotlib.use('Agg')  # Use non-interactive backend
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize, LinearSegmentedColormap
import seaborn as sns
from folium.plugins import MarkerCluster, FeatureGroupSubGroup
import branca.colormap as cm
from datetime import datetime
import math
import glob
import folium

###########################################
# Configuration Parameters
###########################################

# Define runtime configuration
RUNTIME_CONFIG = {
    'DATA_FILES_PATTERN': 'JC2024*citibiketripdata.csv',  # Pattern to match all 2024 monthly files
    'DATA_DIR': 'data/citibike',  # Directory containing the Citibike trip data files
    'OUTPUT_DIR': 'results',  # Output directory for results
    'MODE': 'visualize',  # Mode: 'train', 'predict', 'visualize', 'enhanced-viz', 'animate', 'all'
    'USE_GPU': True,  # Use GPU if available
    'MAX_STATIONS': 50,  # Maximum number of stations to include
    'TRAINING_EPOCHS': 100,  # Number of training epochs
    'SAMPLE_RATE': 1,  # Sampling rate for data loading - reduced to handle more data
    'DISTANCE_THRESHOLD': 2.0,  # Distance threshold for adjacency (km)
    'ANALYZE_SEASONAL_PATTERNS': True,  # Enable seasonal pattern analysis
}

# Citibike configuration parameters
CITIBIKE_CONFIG = {
    # Dataset characteristics
    'num_stations': RUNTIME_CONFIG['MAX_STATIONS'],
    'time_periods': 24,  # Hours of the day
    'day_types': 2,  # Weekday and weekend
    'user_types': 2,  # Member and casual riders
    
    # Architecture - Memory efficiency settings
    'gcn_hidden_dim': 128,  # Hidden dimensions for GCN
    'spatial_hidden_dim': 16,  # Hidden dimension for spatial features
    'temporal_hidden_dim': 24,  # Hidden dimension for temporal features
    'latent_dim': 64,  # Latent dimension for the model
    'dropout_rate': 0.3,  # Dropout rate to prevent overfitting
    'l2_reg': 1e-5,  # L2 regularization
    
    # Training parameters
    'batch_size': 64,  # Batch size
    'epochs': RUNTIME_CONFIG['TRAINING_EPOCHS'],
    'learning_rate': 1e-3,  # Initial learning rate
    'lr_decay_factor': 0.5,  # Learning rate decay factor
    'lr_patience': 5,  # Patience for learning rate scheduler
    'early_stopping_patience': 15,  # Early stopping patience
    
    # Data paths
    'data_path': RUNTIME_CONFIG['DATA_DIR'],
    'output_path': RUNTIME_CONFIG['OUTPUT_DIR'],
    
    # Temporal features
    'peak_morning_start': 7,  # Morning peak hour start (7 AM)
    'peak_morning_end': 10,  # Morning peak hour end (10 AM)
    'peak_evening_start': 16,  # Evening peak hour start (4 PM)
    'peak_evening_end': 19,  # Evening peak hour end (7 PM)
    
    # Trip characteristics
    'avg_trip_duration': 7.39,  # Average trip duration in minutes
    'median_trip_duration': 5.37,  # Median trip duration in minutes
    'avg_station_distance': 3.77,  # Average distance between stations in km
    
    # Feature selection
    'use_distance_features': True,  # Use station distances
    'use_temporal_features': True,  # Use time of day, day of week
    'enable_attention': False,  # Disable attention to save memory
    
    # Visualization settings
    'max_stations_viz': 50,  # Max stations for visualizations
    'sample_rate_viz': RUNTIME_CONFIG['SAMPLE_RATE'],  # Sampling rate for visualizations
    'animation_max_stations': 12,  # Max stations for animations
    'animation_sample_rate': 0.1,  # Sampling rate for animations
    
    # Seasonal analysis
    'analyze_seasons': RUNTIME_CONFIG['ANALYZE_SEASONAL_PATTERNS'],  # Enable seasonal trend analysis
    'seasons': {
        'winter': [1, 2, 12],  # Jan, Feb, Dec
        'spring': [3, 4, 5],  # Mar, Apr, May
        'summer': [6, 7, 8],  # Jun, Jul, Aug
        'fall': [9, 10, 11]  # Sep, Oct, Nov
    },
    
    # Weekend vs Weekday analysis
    'analyze_day_types': True,  # Enable weekday/weekend comparison
}

###########################################
# Helper Functions
###########################################

def create_bezier_curve(lon1, lat1, lon2, lat2, num_points=20):
    """Create Bezier curve coordinates between two points"""
    # Calculate intermediate control point
    control_lon = (lon1 + lon2) / 2 + (lat2 - lat1) * 0.2
    control_lat = (lat1 + lat2) / 2 + (lon1 - lon2) * 0.2
    
    t = np.linspace(0, 1, num_points)
    curve = []
    for i in t:
        x = (1 - i)**2 * lon1 + 2 * (1 - i) * i * control_lon + i**2 * lon2
        y = (1 - i)**2 * lat1 + 2 * (1 - i) * i * control_lat + i**2 * lat2
        curve.append((x, y))
    return curve

def create_flow_map(stations_df, flows, output_dir="results", 
                   flow_colormap='RdYlGn', max_line_weight=10):
    """Create interactive flow map with animated Bezier curves"""
    
    if not flows:
        print("No flows to visualize")
        return

    # Calculate max flow for colormap scaling
    max_flow = max(count for _, _, count in flows)
    colormap = cm.LinearColormap(
        colors=['green', 'yellow', 'red'],
        vmin=0,
        vmax=max_flow
    )

    # Calculate map center from stations
    center_lat = stations_df['lat'].mean()
    center_lng = stations_df['lng'].mean()

    # Base map
    m = folium.Map(
        location=[center_lat, center_lng],
        zoom_start=13,
        tiles='cartodbpositron',
        attr='© OpenStreetMap contributors | © CARTO'
    )

    # Add flow lines
    for origin, destination, count in flows:
        orig = stations_df[stations_df['name'] == origin].iloc[0]
        dest = stations_df[stations_df['name'] == destination].iloc[0]
        
        # Create Bezier curve
        curve_points = create_bezier_curve(orig['lng'], orig['lat'],
                                         dest['lng'], dest['lat'])
        
        folium.PolyLine(
            locations=[[lat, lng] for lng, lat in curve_points],
            color=colormap(count),
            weight=max_line_weight * (count/max_flow),
            opacity=0.7,
            tooltip=f"{origin} to {destination}: {count} trips"
        ).add_to(m)

    # Add colormap to map
    colormap.caption = 'Number of Trips'
    colormap.add_to(m)

    # Save map
    os.makedirs(output_dir, exist_ok=True)
    map_path = os.path.join(output_dir, "flow_map.html")
    m.save(map_path)
    print(f"Flow map saved to {map_path}")
    return m

def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculate haversine distance between two points in kilometers"""
    # Convert decimal degrees to radians
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    
    # Haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c = 2 * math.asin(math.sqrt(a))
    r = 6371  # Radius of earth in kilometers
    
    return c * r

def create_temporal_features(hour, day_of_week, month, config):
    """Create temporal features for the model"""
    # Cyclical encoding for hour of day (sine and cosine)
    hour_sin = np.sin(2 * np.pi * hour / 24)
    hour_cos = np.cos(2 * np.pi * hour / 24)
    
    # Cyclical encoding for day of week
    day_sin = np.sin(2 * np.pi * day_of_week / 7)
    day_cos = np.cos(2 * np.pi * day_of_week / 7)
    
    # Cyclical encoding for month of year
    month_sin = np.sin(2 * np.pi * month / 12)
    month_cos = np.cos(2 * np.pi * month / 12)
    
    # Peak hour indicators
    is_morning_peak = 1 if config['peak_morning_start'] <= hour < config['peak_morning_end'] else 0
    is_evening_peak = 1 if config['peak_evening_start'] <= hour < config['peak_evening_end'] else 0
    
    # Weekend indicator
    is_weekend = 1 if day_of_week >= 5 else 0  # 5=Saturday, 6=Sunday
    
    # Seasonal indicators (using month)
    seasons = config.get('seasons', {})
    is_winter = 1 if month in seasons.get('winter', [12, 1, 2]) else 0
    is_spring = 1 if month in seasons.get('spring', [3, 4, 5]) else 0
    is_summer = 1 if month in seasons.get('summer', [6, 7, 8]) else 0
    is_fall = 1 if month in seasons.get('fall', [9, 10, 11]) else 0
    
    return [hour_sin, hour_cos, day_sin, day_cos, month_sin, month_cos,
            is_morning_peak, is_evening_peak, is_weekend,
            is_winter, is_spring, is_summer, is_fall]

def prepare_adjacency_matrix(distance_matrix, threshold=2.0):
    """Prepare adjacency matrix for GCN based on station distances"""
    num_stations = distance_matrix.shape[0]
    adj_matrix = np.zeros((num_stations, num_stations))
    
    # Create adjacency based on distance threshold
    for i in range(num_stations):
        for j in range(num_stations):
            if i != j and distance_matrix[i, j] <= threshold:
                adj_matrix[i, j] = 1
    
    # Add self-loops
    adj_matrix = adj_matrix + np.eye(num_stations)
    
    # Normalize adjacency matrix
    rowsum = np.array(adj_matrix.sum(1))
    d_inv_sqrt = np.power(rowsum, -0.5).flatten()
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.
    d_mat_inv_sqrt = np.diag(d_inv_sqrt)
    
    normalized_adj = adj_matrix.dot(d_mat_inv_sqrt).transpose().dot(d_mat_inv_sqrt)
    
    return normalized_adj

def load_citibike_data(data_files_pattern, sample_rate=0.2):
    """Load and preprocess Citibike data from multiple files"""
    print(f"Loading Citibike data with {sample_rate*100:.1f}% sampling rate")
    
    # Find all matching files
    data_dir = os.path.dirname(data_files_pattern)
    if not data_dir:
        data_dir = '.'
    file_list = glob.glob(os.path.join(data_dir, os.path.basename(data_files_pattern)))
    
    if not file_list:
        raise FileNotFoundError(f"No data files found matching pattern: {data_files_pattern}")
    
    file_list.sort()  # Ensure files are processed in order
    print(f"Found {len(file_list)} files to process")
    
    # Load data from each file
    all_trips = []
    
    for file_path in file_list:
        file_name = os.path.basename(file_path)
        print(f"Processing file: {file_name}")
        
        # Extract month from filename (assuming format JC2024MMcitibiketripdata.csv)
        try:
            month = int(file_name[6:8])
            print(f"Extracted month: {month}")
        except (ValueError, IndexError):
            month = 0
            print(f"Could not extract month from filename: {file_name}, using placeholder")
        
        # Load data in chunks to save memory
        chunks = []
        
        for chunk in pd.read_csv(file_path, chunksize=10000):
            # Sample the chunk to reduce memory usage
            sampled_chunk = chunk.sample(frac=sample_rate, random_state=42)
            chunks.append(sampled_chunk)
        
        # Combine chunks for this month
        if chunks:
            monthly_trips = pd.concat(chunks)
            print(f"Loaded {len(monthly_trips)} sampled trips from {file_name}")
            
            # Add month column if not already present
            if 'month' not in monthly_trips.columns:
                monthly_trips['month'] = month
            
            all_trips.append(monthly_trips)
        else:
            print(f"No data loaded from {file_name}")
    
    # Combine all monthly data
    if not all_trips:
        raise ValueError("No trip data was loaded from any file")
    
    trips_df = pd.concat(all_trips)
    print(f"Combined dataset contains {len(trips_df)} trips")
    
    # Basic data cleaning
    # Remove trips with missing station information
    trips_df = trips_df.dropna(subset=['start_station_name', 'end_station_name', 
                                       'start_lat', 'start_lng', 
                                       'end_lat', 'end_lng'])
    
    # Convert timestamps to datetime - use format='mixed' to handle various timestamp formats
    print("Converting timestamps to datetime...")
    trips_df['started_at'] = pd.to_datetime(trips_df['started_at'], format='mixed', errors='coerce')
    trips_df['ended_at'] = pd.to_datetime(trips_df['ended_at'], format='mixed', errors='coerce')
    
    # Extract time features
    trips_df['start_hour'] = trips_df['started_at'].dt.hour
    trips_df['start_day'] = trips_df['started_at'].dt.dayofweek  # Monday=0, Sunday=6
    trips_df['is_weekend'] = trips_df['start_day'].apply(lambda x: 1 if x >= 5 else 0)
    
    # Make sure month is available
    if 'month' not in trips_df.columns:
        trips_df['month'] = trips_df['started_at'].dt.month
    
    # Calculate trip duration in minutes
    trips_df['duration_minutes'] = (trips_df['ended_at'] - trips_df['started_at']).dt.total_seconds() / 60
    
    # Filter out unrealistic trips (e.g., maintenance-related trips or errors)
    trips_df = trips_df[(trips_df['duration_minutes'] > 0) & 
                        (trips_df['duration_minutes'] < 24*60)]  # Less than 24 hours
    
    print(f"After cleaning: {len(trips_df)} trips")
    
    return trips_df

def get_top_stations(trips_df, max_stations):
    """Get the top stations by trip count and create station mapping"""
    # Get unique stations
    start_counts = trips_df['start_station_name'].value_counts()
    end_counts = trips_df['end_station_name'].value_counts()
    combined_counts = start_counts.add(end_counts, fill_value=0)
    
    # Take the top stations
    max_stations = min(max_stations, len(combined_counts))
    top_stations = combined_counts.sort_values(ascending=False).head(max_stations).index.tolist()
    
    print(f"Using top {len(top_stations)} stations")
    
    # Create station mapping
    station_mapping = {station: idx for idx, station in enumerate(top_stations)}
    
    # Create station data dictionary
    station_data = {}
    
    for station in top_stations:
        # Get station coordinates from the first occurrence
        start_rows = trips_df[trips_df['start_station_name'] == station]
        end_rows = trips_df[trips_df['end_station_name'] == station]
        
        if not start_rows.empty:
            lat = start_rows['start_lat'].iloc[0]
            lng = start_rows['start_lng'].iloc[0]
        elif not end_rows.empty:
            lat = end_rows['end_lat'].iloc[0]
            lng = end_rows['end_lng'].iloc[0]
        else:
            lat, lng = 0, 0
            print(f"No coordinate data found for station: {station}")
        
        # Calculate station popularity
        popularity = (
            trips_df['start_station_name'].value_counts().get(station, 0) +
            trips_df['end_station_name'].value_counts().get(station, 0)
        )
        
        station_data[station_mapping[station]] = {
            'name': station,
            'lat': lat,
            'lng': lng,
            'popularity': popularity
        }
    
    return station_mapping, station_data, top_stations

def calculate_distance_matrix(station_data):
    """Calculate distance matrix between stations"""
    num_stations = len(station_data)
    distance_matrix = np.zeros((num_stations, num_stations))
    
    for i in range(num_stations):
        for j in range(num_stations):
            if i != j:
                station_i = station_data[i]
                station_j = station_data[j]
                
                # Calculate haversine distance
                distance_matrix[i, j] = haversine_distance(
                    station_i['lat'], station_i['lng'],
                    station_j['lat'], station_j['lng']
                )
    
    return distance_matrix

def create_od_matrices(trips_df, station_mapping, num_stations):
    """Create origin-destination matrices for different time periods"""
    # Create 4D matrix: [month, day_type, hour, origin, destination]
    hours = 24
    day_types = 2  # Weekday and weekend
    months = 12  # All months of the year
    
    od_matrices = np.zeros((months, day_types, hours, num_stations, num_stations))
    
    # Filter trips to only include mapped stations
    mapped_trips = trips_df[
        trips_df['start_station_name'].isin(station_mapping.keys()) & 
        trips_df['end_station_name'].isin(station_mapping.keys())
    ]
    
    # Group trips by month, day type, hour, origin, and destination
    for month in range(1, months+1):
        month_trips = mapped_trips[mapped_trips['month'] == month]
        
        if month_trips.empty:
            print(f"No trips found for month {month}")
            continue
        
        print(f"Processing month {month} with {len(month_trips)} trips")
        
        for day_type in range(day_types):  # 0=weekday, 1=weekend
            is_weekend = day_type  # 0=False, 1=True
            
            day_trips = month_trips[month_trips['is_weekend'] == is_weekend]
            
            for hour in range(hours):
                hour_trips = day_trips[day_trips['start_hour'] == hour]
                
                for _, trip in hour_trips.iterrows():
                    origin_idx = station_mapping.get(trip['start_station_name'])
                    dest_idx = station_mapping.get(trip['end_station_name'])
                    
                    if origin_idx is not None and dest_idx is not None:
                        od_matrices[month-1, day_type, hour, origin_idx, dest_idx] += 1
    
    print(f"Created OD matrices of shape {od_matrices.shape}")
    print(f"Total trips in OD matrices: {np.sum(od_matrices)}")
    
    return od_matrices

def create_features_and_targets(od_matrices, station_data, distance_matrix, config):
    """Create features and targets for the model"""
    months, day_types, hours, num_stations, _ = od_matrices.shape
    
    features = []
    targets = []
    
    # Process data for each month
    for month in range(months):
        for day_type in range(day_types):
            for hour in range(hours):
                # Process every other hour for memory efficiency
                if hour % 2 == 0:
                    for origin in range(num_stations):
                        for dest in range(num_stations):
                            # Skip self-loops
                            if origin == dest:
                                continue
                            
                            # Get the flow value (target)
                            flow = od_matrices[month, day_type, hour, origin, dest]
                            
                            # Only include non-zero flows to reduce dataset size
                            if flow > 0:
                                # 1. Origin station attributes
                                origin_features = [
                                    station_data[origin]['lat'],
                                    station_data[origin]['lng'],
                                    station_data[origin]['popularity'] / 1000,
                                    1,  # Is origin indicator
                                    0   # Is destination indicator
                                ]
                                
                                # 2. Destination station attributes
                                dest_features = [
                                    station_data[dest]['lat'],
                                    station_data[dest]['lng'],
                                    station_data[dest]['popularity'] / 1000,
                                    0,  # Is origin indicator
                                    1   # Is destination indicator
                                ]
                                
                                # 3. All stations attributes (simplified to top 5 stations by distance)
                                station_features = []
                                
                                # Get distances from origin to all stations
                                distances = [(i, distance_matrix[origin, i]) 
                                            for i in range(num_stations) if i != origin and i != dest]
                                # Sort by distance and take top 5
                                top_stations = sorted(distances, key=lambda x: x[1])[:min(5, len(distances))]
                                
                                for station_idx, dist in top_stations:
                                    station_features.extend([
                                        station_data[station_idx]['lat'],
                                        station_data[station_idx]['lng'],
                                        station_data[station_idx]['popularity'] / 1000,
                                        dist
                                    ])
                                
                                # Pad if we have fewer than 5 stations
                                while len(top_stations) < 5:
                                    station_features.extend([0, 0, 0, 0])
                                    top_stations.append((None, None))
                                
                                # 4. Spatial features
                                spatial_features = [
                                    distance_matrix[origin, dest],  # Distance between origin and destination
                                    station_data[origin]['lat'] - station_data[dest]['lat'],  # Lat difference
                                    station_data[origin]['lng'] - station_data[dest]['lng'],  # Lng difference
                                ]
                                
                                # 5. Temporal features
                                temp_features = create_temporal_features(hour, day_type*5, month+1, config)
                                
                                # Combine all features
                                all_features = (
                                    origin_features + 
                                    dest_features + 
                                    station_features + 
                                    spatial_features + 
                                    temp_features
                                )
                                
                                features.append(all_features)
                                targets.append(flow)
    
    return np.array(features), np.array(targets)

###########################################
# Dataset and Model Classes
###########################################

class CitibikeODDataset(Dataset):
    """Custom Dataset for Citibike Origin-Destination Matrix data"""
    
    def __init__(self, features, targets):
        self.features = torch.FloatTensor(features)
        self.targets = torch.FloatTensor(targets)
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        return self.features[idx], self.targets[idx]

class GCNLayer(nn.Module):
    """Graph Convolutional Network Layer"""
    
    def __init__(self, in_features, out_features, activation=F.relu, dropout_rate=0.3):
        super(GCNLayer, self).__init__()
        self.linear = nn.Linear(in_features, out_features, bias=False)
        self.activation = activation
        self.dropout = nn.Dropout(dropout_rate) if dropout_rate > 0 else None
    
    def forward(self, inputs, adj_matrix):
        # Linear transformation
        hidden = self.linear(inputs)
        
        # Apply graph convolution: each node aggregates features from its neighbors
        batch_size, num_nodes, _ = hidden.shape
        
        for i in range(batch_size):
            hidden[i] = torch.matmul(adj_matrix, hidden[i])
        
        # Apply activation function
        if self.activation is not None:
            hidden = self.activation(hidden)
        
        # Apply dropout
        if self.dropout is not None:
            hidden = self.dropout(hidden)
        
        return hidden

class CitibikeODModel(nn.Module):
    """Model for Citibike Origin-Destination Matrix Estimation"""
    
    def __init__(self, config, feature_dim, normalized_adj):
        super(CitibikeODModel, self).__init__()
        
        self.config = config
        self.feature_dim = feature_dim
        
        # Register normalized adjacency matrix as buffer
        self.register_buffer('normalized_adj', torch.FloatTensor(normalized_adj))
        
        # Dimensions
        num_stations = config['num_stations']
        gcn_hidden_dim = config['gcn_hidden_dim']
        spatial_hidden_dim = config['spatial_hidden_dim']
        temporal_hidden_dim = config['temporal_hidden_dim']
        latent_dim = config['latent_dim']
        
        # Divide features into categories
        self.feature_categories = {
            'station_features': int(feature_dim * 0.5),  # 50% for station features
            'spatial_features': int(feature_dim * 0.2),  # 20% for spatial features
            'temporal_features': feature_dim - int(feature_dim * 0.5) - int(feature_dim * 0.2)  # 30% for temporal features
        }
        
        # Feature encoders
        self.station_encoder = nn.Sequential(
            nn.Linear(self.feature_categories['station_features'], gcn_hidden_dim),
            nn.ReLU(),
            nn.Dropout(config['dropout_rate'])
        )
        
        self.spatial_encoder = nn.Sequential(
            nn.Linear(self.feature_categories['spatial_features'], spatial_hidden_dim),
            nn.ReLU(),
            nn.Dropout(config['dropout_rate'])
        )
        
        self.temporal_encoder = nn.Sequential(
            nn.Linear(self.feature_categories['temporal_features'], temporal_hidden_dim),
            nn.ReLU(),
            nn.Dropout(config['dropout_rate'])
        )
        
        # GCN for station relationships
        self.gcn = GCNLayer(
            in_features=gcn_hidden_dim,
            out_features=gcn_hidden_dim,
            dropout_rate=config['dropout_rate']
        )
        
        # Feature fusion - concatenate encoded features
        self.fusion_dim = gcn_hidden_dim + spatial_hidden_dim + temporal_hidden_dim
        
        # MLP for final prediction
        self.mlp = nn.Sequential(
            nn.Linear(self.fusion_dim, latent_dim),
            nn.ReLU(),
            nn.Dropout(config['dropout_rate']),
            nn.Linear(latent_dim, latent_dim // 2),
            nn.ReLU(),
            nn.Dropout(config['dropout_rate']),
            nn.Linear(latent_dim // 2, 1)
        )
    
    def forward(self, x):
        batch_size = x.shape[0]
        
        # Split input by feature categories
        spatial_features = x[:, self.feature_categories['station_features']:
                            self.feature_categories['station_features'] + 
                            self.feature_categories['spatial_features']]
        temporal_features = x[:, self.feature_categories['station_features'] + 
                           self.feature_categories['spatial_features']:]
        
        # Encode features
        encoded_station = self.station_encoder(station_features)
        encoded_spatial = self.spatial_encoder(spatial_features)
        encoded_temporal = self.temporal_encoder(temporal_features)
        
        # Reshape station features for GCN (memory-efficient version)
        encoded_station_reshaped = encoded_station.unsqueeze(1)  # [batch_size, 1, gcn_hidden_dim]
        
        # Apply GCN - simplified to avoid memory issues
        gcn_out = self.gcn(encoded_station_reshaped, self.normalized_adj[:1, :1])
        gcn_out = gcn_out.squeeze(1)  # [batch_size, gcn_hidden_dim]
        
        # Concatenate all encoded features
        fusion = torch.cat([gcn_out, encoded_spatial, encoded_temporal], dim=1)
        
        # Final prediction
        output = self.mlp(fusion)
        
        return output.squeeze()

def create_static_flow_map(stations_df, od_matrix, top_n_flows=50, output_dir="results/flow_visualizations", 
                          period_name="All Day", cmap='viridis'):
    """Create a static flow map visualization for a given time period"""
    print(f"Generating visualization for {period_name}")
    
    # Prepare a dataframe of flows
    flows = []
    
    # Get top flows from the OD matrix
    for origin_idx in range(od_matrix.shape[0]):
        for dest_idx in range(od_matrix.shape[1]):
            if origin_idx != dest_idx:  # Skip self-loops
                flow_count = od_matrix[origin_idx, dest_idx]
                if flow_count > 0:
                    origin_name = stations_df.iloc[origin_idx]['name']
                    dest_name = stations_df.iloc[dest_idx]['name']
                    flows.append((origin_name, dest_name, flow_count))
    
    # Sort flows by count (descending)
    flows = sorted(flows, key=lambda x: x[2], reverse=True)
    
    # Take top N flows
    flows = flows[:top_n_flows]
    
    print(f" Number of trips in this period: {sum([f[2] for f in flows])}")
    print(f" OD matrix shape: {od_matrix.shape}")
    
    # Create flow map
    fig, ax = plt.figure(figsize=(12, 10)), plt.gca()
    
    # Set background color
    ax.set_facecolor('#f8f8f8')
    
    # Plot stations
    stations_x = stations_df['lng'].values
    stations_y = stations_df['lat'].values
    ax.scatter(stations_x, stations_y, c='black', s=20, alpha=0.7, zorder=3)
    
    # Create colormap
    min_flow = min([flow[2] for flow in flows]) if flows else 0
    max_flow = max([flow[2] for flow in flows]) if flows else 1
    norm = Normalize(vmin=min_flow, vmax=max_flow)
    flow_cmap = plt.get_cmap(cmap)
    
    # Plot flows as curved arrows
    lines = []
    colors = []
    widths = []
    
    for origin, destination, count in flows:
        orig_station = stations_df[stations_df['name'] == origin].iloc[0]
        dest_station = stations_df[stations_df['name'] == destination].iloc[0]
        
        # Create a curved line
        curve_points = create_bezier_curve(
            orig_station['lng'], orig_station['lat'],
            dest_station['lng'], dest_station['lat']
        )
        
        lines.append(curve_points)
        colors.append(flow_cmap(norm(count)))
        widths.append(1 + 5 * (count - min_flow) / (max_flow - min_flow + 1e-10))
    
    # Add the lines to the plot
    lc = LineCollection(lines, colors=colors, linewidths=widths, alpha=0.7, zorder=2)
    ax.add_collection(lc)
    
    # Add a colorbar
    sm = plt.cm.ScalarMappable(cmap=flow_cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, orientation='vertical', shrink=0.7)
    cbar.set_label('Number of Trips')
    
    # Set map bounds
    ax.set_xlim(stations_x.min() - 0.01, stations_x.max() + 0.01)
    ax.set_ylim(stations_y.min() - 0.01, stations_y.max() + 0.01)
    
    # Set title and labels
    ax.set_title(f'Bike Trip Flows: {period_name}', fontsize=16)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Save the map
    map_path = os.path.join(output_dir, f"{period_name.lower().replace(' ', '_')}_flow_map.png")
    plt.savefig(map_path, dpi=300, bbox_inches='tight')
    print(f" Flow map saved to {map_path}")
    
    # Save as SVG too
    svg_path = os.path.join(output_dir, f"{period_name.lower().replace(' ', '_')}_flow_map.svg")
    plt.savefig(svg_path, format='svg', bbox_inches='tight')
    print(f" Flow map saved as SVG: {svg_path}")
    
    plt.close()
    
    return map_path

def create_flow_visuals(trips_df, stations_df, station_mapping, output_dir="results"):
    """Create flow visualization maps and OD matrix heatmaps for different time periods"""
    print("Generating flow maps and OD matrix heatmaps...")
    
    # Define time periods for analysis
    time_periods = {
        "Morning Peak (7-10 AM)": (7, 10),
        "Afternoon (10 AM-4 PM)": (10, 16),
        "Evening Peak (4-7 PM)": (16, 19),
        "All Day": (0, 24)
    }
    
    # Create output directories
    flow_viz_dir = os.path.join(output_dir, "flow_visualizations")
    od_matrix_dir = os.path.join(output_dir, "od_matrix_heatmaps")
    os.makedirs(flow_viz_dir, exist_ok=True)
    os.makedirs(od_matrix_dir, exist_ok=True)
    
    # Process each time period
    for period_name, (start_hour, end_hour) in time_periods.items():
        # Filter trips for this time period
        if start_hour < end_hour:
            period_trips = trips_df[(trips_df['start_hour'] >= start_hour) & 
                                    (trips_df['start_hour'] < end_hour)]
        else:
            # Handle overnight periods (e.g., night 10 PM - 5 AM)
            period_trips = trips_df[(trips_df['start_hour'] >= start_hour) | 
                                    (trips_df['start_hour'] < end_hour)]
        
        # Create OD matrix for this period
        num_stations = len(station_mapping)
        od_matrix = np.zeros((num_stations, num_stations))
        
        for _, trip in period_trips.iterrows():
            origin = station_mapping.get(trip['start_station_name'])
            destination = station_mapping.get(trip['end_station_name'])
            
            if origin is not None and destination is not None:
                od_matrix[origin, destination] += 1
        
        # Generate static flow map
        create_static_flow_map(
            stations_df, 
            od_matrix, 
            top_n_flows=50,
            output_dir=flow_viz_dir,
            period_name=period_name
        )
        
        # Create OD matrix heatmap
        station_names = [stations_df.iloc[i]['name'] for i in range(num_stations)]
        od_df = pd.DataFrame(od_matrix, index=station_names, columns=station_names)
        
        create_od_matrix_heatmap(
            od_df,
            period_name,
            od_matrix_dir,
            description=f"Total flows: {int(od_matrix.sum())}"
        )
    
    print(f"Flow maps saved to {flow_viz_dir}")
    print(f"OD matrix heatmaps saved to {od_matrix_dir}")
    
    return flow_viz_dir, od_matrix_dir

###########################################
# Main Function
###########################################

def main():
    """Main function to orchestrate the entire workflow"""
    try:
        # Start with runtime parameters
        mode = RUNTIME_CONFIG['MODE']
        data_dir = RUNTIME_CONFIG['DATA_DIR']
        data_files_pattern = os.path.join(data_dir, RUNTIME_CONFIG['DATA_FILES_PATTERN'])
        output_dir = RUNTIME_CONFIG['OUTPUT_DIR']
        max_stations = RUNTIME_CONFIG['MAX_STATIONS']
        sample_rate = RUNTIME_CONFIG['SAMPLE_RATE']
        
        # Determine device for PyTorch
        device = 'cuda' if RUNTIME_CONFIG['USE_GPU'] and torch.cuda.is_available() else 'cpu'
        
        # Print analysis parameters
        print("Starting analysis with parameters:")
        print({
            'mode': mode,
            'data_dir': data_dir,
            'output_dir': output_dir,
            'max_stations': max_stations,
            'sample_rate': sample_rate
        })
        
        # Make sure output directory exists
        os.makedirs(output_dir, exist_ok=True)
        
        # Load data
        trips_df = load_citibike_data(data_files_pattern, sample_rate=sample_rate)
        
        # Get top stations
        station_mapping, station_data, top_stations = get_top_stations(trips_df, max_stations)
        
        # Create a dataframe from station_data for easy access
        stations_df = pd.DataFrame([
            {
                'name': data['name'],
                'lat': data['lat'],
                'lng': data['lng'],
                'popularity': data['popularity']
            }
            for idx, data in station_data.items()
        ])
        
        # Different modes
        if mode in ['visualize', 'enhanced-viz', 'all']:
            print("\nGenerating core visualizations...")
            
            # Create interactive station map
            interactive_station_map(
                stations_df,
                map_title="Citibike Stations",
                output_dir=output_dir
            )
            
            # Create flow visualizations
            create_flow_visuals(trips_df, stations_df, station_mapping, output_dir)
            
            # Add interactive flow map with folium (fixed version)
            period_trips = trips_df  # Use all trips
            flows = []
            
            for _, trip in period_trips.iterrows():
                origin = trip['start_station_name']
                destination = trip['end_station_name']
                
                if origin in station_mapping and destination in station_mapping:
                    flows.append((origin, destination, 1))
            
            # Aggregate flows
            flow_counts = {}
            for origin, dest, count in flows:
                key = (origin, dest)
                flow_counts[key] = flow_counts.get(key, 0) + count
            
            # Convert to list of tuples
            aggregated_flows = [(origin, dest, count) 
                               for (origin, dest), count in flow_counts.items()]
            
            # Take top flows
            top_flows = sorted(aggregated_flows, key=lambda x: x[2], reverse=True)[:100]
            
            # Create interactive flow map
            create_flow_map(stations_df, top_flows, output_dir=output_dir)
            
        if mode in ['train', 'all']:
            # Calculate distance matrix
            distance_matrix = calculate_distance_matrix(station_data)
            
            # Create OD matrices
            od_matrices = create_od_matrices(trips_df, station_mapping, len(station_mapping))
            
            # Create features and targets
            features, targets = create_features_and_targets(od_matrices, station_data, distance_matrix, CITIBIKE_CONFIG)
            
            # Split data
            X_train, X_test, y_train, y_test = train_test_split(
                features, targets, test_size=0.2, random_state=42
            )
            
            # Normalize features
            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)
            
            # Prepare adjacency matrix
            normalized_adj = prepare_adjacency_matrix(
                distance_matrix, 
                threshold=RUNTIME_CONFIG['DISTANCE_THRESHOLD']
            )
            
            # Split validation set
            X_train, X_val, y_train, y_val = train_test_split(
                X_train, y_train, test_size=0.2, random_state=42
            )
            
            # Initialize model
            model = CitibikeODModel(
                CITIBIKE_CONFIG,
                features.shape[1],
                normalized_adj
            ).to(device)
            
            # Train model
            trained_model, history = train_model(
                model, X_train, y_train, X_val, y_val, CITIBIKE_CONFIG, device
            )
            
            # Evaluate model
            metrics = evaluate_model(trained_model, X_test, y_test, device)
            
            # Extended evaluation
            extended_evaluate_model(
                trained_model, X_test, y_test, 
                station_data, station_mapping, distance_matrix,
                trips_df, CITIBIKE_CONFIG, device
            )
            
            # Visualize results
            plot_training_history(history, output_dir)
            visualize_predictions(trained_model, X_test, y_test, output_dir, device=device)
        
        if mode in ['enhanced-viz', 'all'] and CITIBIKE_CONFIG['analyze_seasons']:
            # Perform seasonal analysis
            analyze_seasonal_patterns(trips_df, output_dir, CITIBIKE_CONFIG)
            
            # Analyze day type patterns
            if CITIBIKE_CONFIG['analyze_day_types']:
                analyze_day_type_patterns(trips_df, output_dir, CITIBIKE_CONFIG)
        
        print("\nAnalysis completed successfully!")
        
    except Exception as e:
        print(f"\nError in main execution: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()


Starting analysis with parameters:
{'mode': 'visualize', 'data_dir': 'data/citibike', 'output_dir': 'results', 'max_stations': 50, 'sample_rate': 1}
Loading Citibike data with 100.0% sampling rate
Found 12 files to process
Processing file: JC202401citibiketripdata.csv
Extracted month: 1
Loaded 50661 sampled trips from JC202401citibiketripdata.csv
Processing file: JC202402citibiketripdata.csv
Extracted month: 2
Loaded 55613 sampled trips from JC202402citibiketripdata.csv
Processing file: JC202403citibiketripdata.csv
Extracted month: 3
Loaded 65581 sampled trips from JC202403citibiketripdata.csv
Processing file: JC202404citibiketripdata.csv
Extracted month: 4
Loaded 79116 sampled trips from JC202404citibiketripdata.csv
Processing file: JC202405citibiketripdata.csv
Extracted month: 5
Loaded 97479 sampled trips from JC202405citibiketripdata.csv
Processing file: JC202406citibiketripdata.csv
Extracted month: 6
Loaded 111115 sampled trips from JC202406citibiketripdata.csv
Processing file: JC2